# Enruta trabajo entre varias llamadas
**Clases complementarias** — *Agentes Seguros con NemoClaw* (Platzi × NVIDIA Deep Learning Institute)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uncrayon/curso-seguridad-agentes-nemoclaw/blob/main/clases/3.1-enrutamiento-y-workflows/3.1-enrutamiento-y-workflows.ipynb)

En el Módulo 2 el modelo decidía cada paso dentro de un loop. Aquí **tu código** decide la forma del trabajo y el modelo solo hace piezas pequeñas. Vas a construir, para el proyecto **Soporte NemoClaw**:

1. Un **router de triage** con un enum fijo: `instalacion | api_key | brev | seguridad | otro`, con un especialista por categoría.
2. **JSON estricto y validación**: qué haces cuando el modelo inventa una etiqueta.
3. **ReWOO**: planear una vez → ejecutar búsquedas en paralelo con `concurrent.futures` → sintetizar una vez.
4. **Fan-out / fan-in** a varios especialistas, con una rama que falla.
5. **Medir la latencia** y decidir cuándo NO dividir.

*Basado en la lección 02a-routing del curso de NVIDIA DLI "Securing Agents with OpenShell and NemoClaw" (Apache-2.0). Adaptación al español para Platzi.*

> **¿Sin API key?** Ejecuta todo igual: el notebook detecta que no hay key y usa un **modelo simulado (MOCK)** que imita la forma de las respuestas de la API. Con tu key `nvapi-` (clase 1.2), las mismas celdas llaman al modelo real.

> **¿No programas?** No necesitas leer el código. Presiona ▶ en cada celda, en orden, y lee el texto y lo que aparece debajo (o usa **Entorno de ejecución → Ejecutar todo** en Colab). Las celdas con ⚙️ son de preparación: se ejecutan igual, pero no necesitas abrirlas. Las palabras nuevas están en el [glosario](https://github.com/uncrayon/curso-seguridad-agentes-nemoclaw/blob/main/GLOSARIO.md).

In [ ]:
# @title ⚙️ Configuración (ejecútala siempre primero)
# --- Configuración (ejecútala siempre primero) ---------------------------------
import os, sys, json, subprocess

def _instala(paquete, modulo=None):
    try:
        __import__(modulo or paquete)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

_instala("openai")
_instala("python-dotenv", "dotenv")

# 1) Google Colab: guarda tu key en Secrets (icono de llave 🔑) con el nombre NVIDIA_API_KEY
try:
    from google.colab import userdata
    os.environ.setdefault("NVIDIA_API_KEY", userdata.get("NVIDIA_API_KEY") or "")
except Exception:
    pass
# 2) Local: archivo .env en la raíz del repo (copia .env.example) o `export NVIDIA_API_KEY=nvapi-...`
from dotenv import load_dotenv
load_dotenv(); load_dotenv("../../.env")

BASE_URL    = os.environ.get("NVIDIA_BASE_URL", "https://integrate.api.nvidia.com/v1")
MODEL       = os.environ.get("NVIDIA_MODEL", "nvidia/nemotron-3.5-lightning-30b-a3b")
EMBED_MODEL = os.environ.get("NVIDIA_EMBED_MODEL", "nvidia/nemotron-3-embed-1b")
API_KEY     = (os.environ.get("NVIDIA_API_KEY") or "").strip()
USE_MOCK    = not API_KEY

if API_KEY and "integrate.api.nvidia.com" in BASE_URL and not API_KEY.startswith("nvapi-"):
    print("⚠  Tu key no empieza con 'nvapi-'. ¿Copiaste la key completa de build.nvidia.com?")
print("Modo:", "MOCK (sin NVIDIA_API_KEY: respuestas simuladas)" if USE_MOCK else f"EN VIVO · {MODEL}")

In [ ]:
# @title ⚙️ Cliente: modelo real o simulado, con errores explicados en español
# --- Cliente: modelo real o simulado, con errores explicados en español --------
import re, time, math, hashlib
from types import SimpleNamespace as NS

def _msg(content):
    m = NS(content=content, tool_calls=None, role="assistant", reasoning_content=None)
    return NS(choices=[NS(message=m, finish_reason="stop")], usage=NS(prompt_tokens=0, completion_tokens=0, total_tokens=0))

def _call(name, args):
    tc = NS(id="call_" + hashlib.md5((name + json.dumps(args)).encode()).hexdigest()[:8], type="function",
            function=NS(name=name, arguments=json.dumps(args, ensure_ascii=False)))
    m = NS(content=None, tool_calls=[tc], role="assistant", reasoning_content=None)
    return NS(choices=[NS(message=m, finish_reason="tool_calls")], usage=NS(prompt_tokens=0, completion_tokens=0, total_tokens=0))

class MockClient:
    """Imita la FORMA de la API de chat completions. No es inteligente: sigue reglas.
    Cada notebook puede agregar reglas propias con  @regla_mock  (ver más abajo)."""
    reglas = []
    def __init__(self):
        self.chat = NS(completions=NS(create=self._create))
        self.embeddings = NS(create=self._embed)
    def _create(self, model=None, messages=None, tools=None, **kw):
        nombres = [t["function"]["name"] for t in (tools or [])]
        for r in self.reglas:
            out = r(messages, nombres)
            if out is not None:
                return out
        ultimo = messages[-1]
        texto = (ultimo.get("content") or "")
        if ultimo["role"] == "tool":
            return _msg("Según la herramienta: " + str(ultimo["content"]))
        if re.search(r"c[oó]mo me llamo|what is my name", texto, re.I):
            vistos = [re.search(r"(?:me llamo|my name is)\s+(\w+)", m.get("content") or "", re.I)
                      for m in messages[:-1] if m["role"] == "user"]
            vistos = [v.group(1) for v in vistos if v]
            return _msg(f"Te llamas {vistos[-1]}." if vistos else "No sé cómo te llamas: no me lo has dicho en esta conversación.")
        return _msg("(mock) Respuesta simulada. Configura NVIDIA_API_KEY para hablar con el modelo real.")
    def _embed(self, model=None, input=None, **kw):
        textos = [input] if isinstance(input, str) else list(input)
        return NS(data=[NS(embedding=_vector_bolsa(t), index=i) for i, t in enumerate(textos)])

def regla_mock(f):
    """Decorador: agrega una regla al modelo simulado. f(messages, nombres_de_tools) -> respuesta | None"""
    MockClient.reglas.append(f)
    return f

_VACIAS = set("que qué como cómo para por con una uno los las del este esta eso esto hago hace tengo puedo sale mis tus sus the and you what how".split())

def _vector_bolsa(texto, dim=256):
    """Embedding simulado: bolsa de palabras con hashing. Sirve para que RAG funcione offline."""
    v = [0.0] * dim
    for w in re.findall(r"\w+", texto.lower()):
        if len(w) > 2 and w not in _VACIAS:
            w = w[:6]                       # raíz burda: "reanudar" ~ "reanudé"
            v[int(hashlib.md5(w.encode()).hexdigest(), 16) % dim] += 1.0
    n = math.sqrt(sum(x * x for x in v)) or 1.0
    return [x / n for x in v]

def diagnostico(e):
    """Traduce un error de la API a: qué pasó + qué hacer. Es el 'FAQ' en código."""
    status = getattr(e, "status_code", None)
    cuerpo = str(getattr(e, "body", "") or e)[:300]
    tabla = {
        400: "400 · Solicitud inválida: algún parámetro no le gusta al modelo (¿tools en un modelo que no las soporta? ¿max_tokens muy alto?). Lee el cuerpo del error.",
        401: "401 · Key rechazada: revisa que NVIDIA_API_KEY sea la key completa (empieza con nvapi-), sin espacios ni comillas. Si la regeneraste, la vieja ya no sirve.",
        403: "403 · Tu key es válida pero no tiene acceso a ese modelo o se acabaron tus créditos. Revisa tu cuenta en build.nvidia.com o prueba otro modelo.",
        404: "404 · Ese modelo o esa ruta no existen. Revisa MODEL / EMBED_MODEL (copia el ID exacto del catálogo) y que BASE_URL termine en /v1.",
        410: "410 · El modelo fue retirado del endpoint gratuito. Cambia MODEL por uno vigente del catálogo de build.nvidia.com.",
        422: "422 · Argumentos con formato incorrecto (p. ej. falta input_type en embeddings o un esquema de tool inválido).",
        429: "429 · Demasiadas solicitudes por minuto. Espera unos segundos y reintenta; no ejecutes celdas en ráfaga.",
    }
    if status in tabla:
        return f"{tabla[status]}\n   Detalle del servidor: {cuerpo}"
    if status and 500 <= status < 600:
        return f"{status} · Falla del lado del servidor o el modelo está arrancando. Reintenta en 30-60 s; si persiste, prueba otro modelo.\n   Detalle: {cuerpo}"
    nombre = type(e).__name__
    if "Timeout" in nombre:
        return "Timeout · El modelo tardó demasiado. Reintenta, baja max_tokens o usa un modelo más rápido."
    if "Connection" in nombre:
        return "Sin conexión · No se pudo llegar a BASE_URL. Revisa tu internet, VPN/proxy corporativo o el valor de BASE_URL."
    return f"{nombre}: {cuerpo}"

if USE_MOCK:
    client = MockClient()
else:
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=60, max_retries=2)

def chat(messages, tools=None, model=None, **kw):
    """Una llamada al modelo. Devuelve la respuesta cruda (choices[0].message, finish_reason, usage)."""
    kw.setdefault("max_tokens", 4096)          # los modelos que razonan gastan tokens antes de responder
    if tools:
        kw["tools"] = tools
    try:
        r = client.chat.completions.create(model=model or MODEL, messages=messages, **kw)
    except Exception as e:
        raise RuntimeError(diagnostico(e)) from None
    ch = r.choices[0]
    if ch.finish_reason == "length" and not (ch.message.content or ch.message.tool_calls):
        print("⚠  finish_reason=length sin contenido: el modelo gastó max_tokens razonando. Sube max_tokens.")
    return r

def embed(textos, input_type="passage", model=None):
    """Embeddings. input_type='query' para preguntas, 'passage' para documentos."""
    textos = [textos] if isinstance(textos, str) else list(textos)
    try:
        r = client.embeddings.create(model=model or EMBED_MODEL, input=textos,
                                     extra_body={"input_type": input_type, "truncate": "END"}) if not USE_MOCK \
            else client.embeddings.create(model=model or EMBED_MODEL, input=textos)
    except Exception as e:
        raise RuntimeError(diagnostico(e)) from None
    return [d.embedding for d in r.data]

print("Cliente listo:", "MockClient" if USE_MOCK else BASE_URL)

In [ ]:
# @title ⚙️ Ayudas de este módulo (ejecútala después de las dos celdas anteriores)
# --- Ayudas de este módulo (ejecútala después de las dos celdas anteriores) -------
import json, re, time
from concurrent.futures import ThreadPoolExecutor, as_completed

LLAMADAS = []          # registro de cada llamada al modelo: (rol, segundos). Así contamos el costo.
LATENCIA_MOCK = 0.3    # sin key simulamos lo que tarda una llamada (en vivo suele ser de 1 a 10 s)

def llamar(rol, mensajes, **kw):
    """Una llamada al modelo, registrada. Devuelve la respuesta completa (choices[0].message...)."""
    t0 = time.time()
    if USE_MOCK:
        time.sleep(LATENCIA_MOCK)
    kw.setdefault("temperature", 0)            # clasificar y planear: queremos respuestas estables
    r = chat(mensajes, **kw)
    LLAMADAS.append((rol, round(time.time() - t0, 2)))
    return r

def llm(rol, usuario, sistema="", **kw):
    """Atajo: system + user -> texto. El rol va en la primera línea del system (y en el registro)."""
    mensajes = [{"role": "system", "content": f"ROL: {rol}\n{sistema}".strip()},
                {"role": "user", "content": usuario}]
    return llamar(rol, mensajes, **kw).choices[0].message.content or ""

def extraer_json(texto):
    """Los modelos a veces envuelven el JSON en prosa o en ```json. Tomamos el primer {...}."""
    m = re.search(r"\{.*\}", texto or "", re.S)
    if not m:
        raise ValueError(f"no encontré JSON en: {(texto or '')[:80]!r}")
    return json.loads(m.group(0))

class ErrorSimulado(Exception):
    """Imita un error HTTP de la API (status_code + body) para practicar el diagnóstico sin gastar llamadas."""
    def __init__(self, status_code, body="(error simulado)"):
        super().__init__(f"HTTP {status_code}")
        self.status_code, self.body = status_code, body

def _rol(messages):
    s = messages[0]["content"] if messages and messages[0]["role"] == "system" else ""
    return s.split("\n")[0].replace("ROL:", "").strip()

def _ultimo_usuario(messages):
    return next((m["content"] for m in reversed(messages) if m["role"] == "user"), "") or ""

def _primeras(texto, n=2):
    """Las primeras n frases de un texto (para respuestas simuladas cortas)."""
    frases = re.split(r"(?<=[.!?])\s+", texto.strip())
    return " ".join(frases[:n])

print("Ayudas listas: llm(), llamar(), extraer_json(), ErrorSimulado")

In [ ]:
# @title ⚙️ Reglas del modelo simulado para ESTE notebook (solo se usan sin API key)
# --- Reglas del modelo simulado para ESTE notebook (solo se usan sin API key) -----
# No necesitas leerlas. Imitan la FORMA de las respuestas (JSON, citas) con palabras clave.
# Ojo: a propósito, el triage simulado INVENTA una etiqueta en un caso, para practicar la validación.

@regla_mock
def _mock_triage(messages, nombres):
    if _rol(messages) != "triage":
        return None
    texto = _ultimo_usuario(messages)
    if texto.startswith("Tu respuesta anterior no es válida"):          # el reintento corrige
        return _msg(json.dumps({"categoria": "otro", "prioridad": "baja",
                                "resumen": "Pregunta administrativa fuera de las categorías técnicas."}, ensure_ascii=False))
    t = texto.lower()
    reglas = [("facturacion", r"reembols|factura|cobro doble"),          # <- etiqueta INVENTADA
              ("seguridad", r"pol[ií]tica|policy|connect tunnel|bloque|permiso|secreto"),
              ("instalacion", r"instal|onboard|docker|wsl|gpu|ready within"),
              ("brev", r"brev|instancia|reanud|launchable|cookie"),
              ("api_key", r"401|403|404|429|key|nvapi")]
    cat = next((c for c, patron in reglas if re.search(patron, t)), "otro")
    prioridad = "alta" if re.search(r"no puedo|no funciona|error|401|403|bloque", t) else "baja"
    salida = json.dumps({"categoria": cat, "prioridad": prioridad,
                         "resumen": texto.strip().split(".")[0][:70]}, ensure_ascii=False)
    return _msg("Claro, esta es la clasificación:\n" + salida if cat == "brev" else salida)

_RESPUESTAS = [   # (categoría, patrón en el ticket, respuesta simulada). Gana la primera que coincide.
    ("instalacion", r"gpu", "No necesitas GPU para los notebooks: corren en Colab o en tu computadora y usan la API de NVIDIA. El laboratorio corre en una instancia de Brev."),
    ("instalacion", r"", "Revisa que Docker esté corriendo con `docker info`. Si el sandbox no quedó listo a tiempo, exporta NEMOCLAW_SANDBOX_READY_TIMEOUT=600, borra el sandbox retenido y repite el onboarding con otro nombre."),
    ("api_key", r"", "Un 401 significa que la API rechazó la key. Comprueba que NVIDIA_API_KEY empiece con nvapi- y no tenga espacios ni comillas. Si la regeneraste, actualízala en todos lados."),
    ("brev", r"deten|apag|stop|archivo", "Usa `brev ls` y luego `brev stop <instancia>`. Detenida no cobra cómputo pero sí disco, y tus datos se conservan, excepto /tmp."),
    ("brev", r"", "Al reanudar Brev, la cookie de acceso del laboratorio suele vencer. Inicia sesión otra vez en el launchable y pega una cookie nueva en Access session."),
    ("seguridad", r"403|connect|bloque|pol[ií]tica", "Ese 403 viene de la política de red del sandbox. Mira la línea DENIED con `nemoclaw <sandbox> logs --tail 50` y agrega el preset con `nemoclaw <sandbox> policy add <preset>`."),
    ("seguridad", r"", "No veo un problema de políticas del sandbox. Un 401 no es un bloqueo de red: eso se vería como CONNECT tunnel failed, response 403."),
    ("otro", r"", "Gracias por escribir. No es un tema técnico que yo pueda resolver: lo paso al equipo del curso."),
    ("generalista", r"", "Puede ser la key o la sesión del laboratorio. Revisa que NVIDIA_API_KEY empiece con nvapi-. Si reanudaste Brev, vuelve a iniciar sesión en el launchable y copia una cookie nueva."),
]

@regla_mock
def _mock_especialista(messages, nombres):
    rol = _rol(messages)
    if not rol.startswith("especialista:"):
        return None
    cat, ticket = rol.split(":", 1)[1], _ultimo_usuario(messages).lower()
    texto = next((r for c, patron, r in _RESPUESTAS if c == cat and re.search(patron, ticket)), "Sin hallazgos.")
    return _msg("(mock) " + texto)

@regla_mock
def _mock_planificador(messages, nombres):
    if _rol(messages) != "planificador":
        return None
    t = _ultimo_usuario(messages).lower()
    candidatas = [(r"deten|paus|apag", "cómo detener la instancia de brev"),
                  (r"archivo|datos", "¿se conservan mis datos si detengo la instancia?"),
                  (r"laboratorio|sesi", "session expired en el laboratorio"),
                  (r"401|key", "qué significa el error 401"),
                  (r"reanud|no responde", "el agente no responde después de reanudar la instancia")]
    consultas = [c for patron, c in candidatas if re.search(patron, t)] or [t[:60]]
    return _msg(json.dumps({"consultas": consultas}, ensure_ascii=False))

@regla_mock
def _mock_sintetizador(messages, nombres):
    if _rol(messages) != "sintetizador":
        return None
    lineas = re.findall(r"^\[(\d+)\] .*? → \[[^\]]+\] (.*)$", _ultimo_usuario(messages), re.M)
    return _msg("(mock) Respuesta:\n" + "\n".join(f"- {_primeras(r, 1)} [{n}]" for n, r in lineas))

@regla_mock
def _mock_fusion(messages, nombres):
    if _rol(messages) != "fusion":
        return None
    t = _ultimo_usuario(messages)
    ok = re.findall(r"^\[(\w+)\] \(mock\) (.+)$", t, re.M)
    faltan = re.findall(r"^\[(\w+)\] NO DISPONIBLE", t, re.M)
    out = "(mock) Respuesta para el estudiante:\n" + "\n".join(f"- {_primeras(c, 1)} (de: {n})" for n, c in ok)
    if faltan:
        out += f"\nOjo: falta la revisión de {', '.join(faltan)}; esta respuesta puede estar incompleta."
    return _msg(out)

print("Reglas mock de la clase 3.1 listas" + ("" if USE_MOCK else " (no se usan: estás en vivo)"))

## 1 · El ticket que no cabe en un solo loop
Un estudiante escribe: *"Me sale 401 en el laboratorio después de reanudar Brev"*. Un loop ReAct podría ir descubriendo qué revisar, llamada por llamada. Pero tú **ya sabes** qué hay que revisar: la key, la instancia de Brev y la sesión del laboratorio. Si la forma del trabajo se conoce antes de empezar, **tu código** puede ser el dueño de esa forma:

| Patrón | Quién decide el siguiente paso | Úsalo cuando |
|---|---|---|
| Loop ReAct (2.2) | El modelo, después de cada observación | El paso 2 depende de lo que salió en el paso 1 |
| Router | Tu código, con una etiqueta que elige el modelo | Cada petición va a **un** especialista |
| ReWOO | El modelo planea **una vez**; tu código ejecuta | Los pasos son independientes entre sí |
| Fan-out | Tu código lanza ramas en paralelo y las une | Varias revisiones independientes del mismo caso |

## 2 · Router de triage con un enum fijo
El modelo **no decide qué hacer**: elige **una etiqueta de una lista cerrada**. Lo demás es Python normal que puedes probar.

Tres reglas:
1. Pide JSON y, si la ruta lo permite, un **esquema** (`response_format` con `json_schema` y `enum`).
2. **Valida en tu código** de todos modos: la salida del modelo es una entrada **no confiable**.
3. Si la etiqueta es inválida: **un** reintento con el error explicado; si vuelve a fallar, `otro` (un humano lo revisa).

In [ ]:
CATEGORIAS  = ["instalacion", "api_key", "brev", "seguridad", "otro"]
PRIORIDADES = ["alta", "media", "baja"]

# El esquema que le pedimos a la API (igual al de la lección de NVIDIA, con nuestras categorías)
ESQUEMA_TRIAGE = {
    "name": "triage", "strict": True,
    "schema": {"type": "object", "additionalProperties": False,
               "properties": {"categoria": {"type": "string", "enum": CATEGORIAS},
                              "prioridad": {"type": "string", "enum": PRIORIDADES},
                              "resumen":   {"type": "string"}},
               "required": ["categoria", "prioridad", "resumen"]},
}

SISTEMA_TRIAGE = f"""Clasificas tickets de estudiantes del curso de NemoClaw.
Categorías (elige exactamente una):
- instalacion: instalar o configurar NemoClaw u OpenShell, Docker, onboarding.
- api_key: la key nvapi- y errores 401/403/404/429 de la API de modelos.
- brev: instancias de Brev, launchable, detener/reanudar, sesión del laboratorio.
- seguridad: políticas del sandbox, red bloqueada, permisos, secretos.
- otro: cualquier otra cosa.
Responde SOLO con JSON: {{"categoria": "...", "prioridad": "alta|media|baja", "resumen": "<una frase>"}}"""

def validar_triage(obj):
    """Devuelve la lista de problemas. Lista vacía = válido."""
    if not isinstance(obj, dict):
        return ["no es un objeto JSON"]
    errores = []
    if obj.get("categoria") not in CATEGORIAS:
        errores.append(f"categoria inválida: {obj.get('categoria')!r}")
    if obj.get("prioridad") not in PRIORIDADES:
        errores.append(f"prioridad inválida: {obj.get('prioridad')!r}")
    if not str(obj.get("resumen", "")).strip():
        errores.append("resumen vacío")
    if set(obj) - {"categoria", "prioridad", "resumen"}:
        errores.append(f"claves de más: {sorted(set(obj) - {'categoria', 'prioridad', 'resumen'})}")
    return errores

print(validar_triage({"categoria": "brev", "prioridad": "alta", "resumen": "401 al reanudar"}))   # []
print(validar_triage({"categoria": "facturacion", "prioridad": "alta", "resumen": "x"}))          # inventada

In [ ]:
def _pedir_triage(mensajes):
    """Pide JSON con esquema. Si la ruta no acepta response_format (400), sigue solo con el prompt."""
    try:
        return llamar("triage", mensajes, response_format={"type": "json_schema", "json_schema": ESQUEMA_TRIAGE})
    except RuntimeError as e:
        if str(e).startswith("400"):
            return llamar("triage", mensajes)
        raise

def clasificar(ticket, ver=False):
    mensajes = [{"role": "system", "content": "ROL: triage\n" + SISTEMA_TRIAGE},
                {"role": "user", "content": ticket}]
    for intento in (1, 2):
        texto = _pedir_triage(mensajes).choices[0].message.content or ""
        try:
            obj = extraer_json(texto)
            errores = validar_triage(obj)
        except (ValueError, json.JSONDecodeError) as e:
            obj, errores = None, [f"JSON ilegible: {e}"]
        if ver:
            print(f"   intento {intento}: {texto.strip()[:80]!r} -> {'OK' if not errores else errores}")
        if not errores:
            return {**obj, "intentos": intento}
        # Reintento: le mostramos su respuesta y el error concreto
        mensajes += [{"role": "assistant", "content": texto},
                     {"role": "user", "content": f"Tu respuesta anterior no es válida ({'; '.join(errores)}). "
                                                 f"Responde SOLO el JSON, con categoria en {CATEGORIAS}."}]
    return {"categoria": "otro", "prioridad": "media", "resumen": ticket[:80], "intentos": 2, "fallback": True}

print(clasificar("Me sale 401 cuando corro el notebook de la clase 1.3", ver=True))

### Especialistas y router
Cada categoría tiene un **especialista**: la misma función con otro *system prompt*. Una pequeña **fábrica** (`crear_especialista`) evita copiar y pegar, como en la lección de NVIDIA. El **router** es un diccionario: código normal, sin modelo.

In [ ]:
def crear_especialista(rol, sistema):
    """Fábrica: devuelve una función ticket -> respuesta con su propio system prompt."""
    def atender(ticket):
        return llm(f"especialista:{rol}", ticket, sistema=sistema)
    atender.rol = rol
    return atender

BASE = "Eres soporte del curso de NemoClaw. Responde en español, en 2-3 frases con pasos concretos. Si no estás seguro, dilo. "
ESPECIALISTAS = {
    "instalacion": crear_especialista("instalacion", BASE + "Tu tema: instalación y onboarding de NemoClaw."),
    "api_key":     crear_especialista("api_key", BASE + "Tu tema: la key nvapi- y los errores de la API de modelos."),
    "brev":        crear_especialista("brev", BASE + "Tu tema: instancias de Brev, launchable y sesión del laboratorio."),
    "seguridad":   crear_especialista("seguridad", BASE + "Tu tema: políticas del sandbox de OpenShell y red bloqueada."),
    "otro":        crear_especialista("otro", "No es un tema técnico. Agradece y di que lo pasas al equipo del curso."),
}

def soporte(ticket, ver=False):
    t = clasificar(ticket, ver=ver)
    especialista = ESPECIALISTAS[t["categoria"]]      # el router: un dict. Nunca falla porque validamos antes.
    return t, especialista(ticket)

TICKETS = [
    "Me sale 401 cuando corro el notebook de la clase 1.3",
    "¿Cómo detengo mi instancia de Brev sin perder mis archivos?",
    "El agente no puede descargar nada: CONNECT tunnel failed, response 403",
    "nemoclaw onboard dice: was created but did not become ready within 180s",
    "Quiero un reembolso, me hicieron un cobro doble",
    "¿Habrá certificado al terminar el curso?",
]
LLAMADAS.clear()
for ticket in TICKETS:
    t, respuesta = soporte(ticket, ver=True)
    marca = " (fallback)" if t.get("fallback") else ""
    print(f"→ {t['categoria']:11s} {t['prioridad']:5s} intentos={t['intentos']}{marca} | {ticket}")
    print(f"   {respuesta[:150]}\n")
print("llamadas al modelo:", len(LLAMADAS))

Fíjate en dos casos:
- **Prosa alrededor del JSON** (`"Claro, esta es la clasificación: {...}"`): `extraer_json` lo rescata y el validador lo acepta.
- **Etiqueta inventada** (`"facturacion"` en el reembolso): el validador la rechaza, el reintento explica el error y el ticket termina en `otro`. **El pipeline nunca se cae** y nunca manda el ticket a un especialista que no existe.

> Con `response_format` y un esquema `strict`, muchas rutas ya impiden etiquetas fuera del enum. **Valida igual**: no todos los modelos ni todos los proveedores lo garantizan, y el día que cambies de modelo tu validador seguirá ahí.

### ✏️ Tu turno
Escribe tres tickets tuyos (uno ambiguo, que toque dos categorías). ¿Qué categoría elige el router? Si un ticket toca dos temas, el router elige **uno**: en la sección 4 verás el patrón para revisar **varios** a la vez.

In [ ]:
MIS_TICKETS = [
    "Mi sandbox no arranca después de instalar Docker en WSL",
    "¿Puedo usar el curso sin tarjeta de crédito?",
    "Reanudé Brev y ahora la política bloquea pypi.org",     # ambiguo: brev + seguridad
]
for ticket in MIS_TICKETS:
    t = clasificar(ticket)
    print(f"{t['categoria']:11s} {t['prioridad']:5s} | {ticket}")

## 3 · ReWOO: planear → ejecutar en paralelo → sintetizar
**ReWOO** (*Reason Without Observation*, Xu et al., 2023) escribe **todo el plan antes** de ejecutar nada. Como las búsquedas no dependen unas de otras, tu código las corre **todas a la vez**. Total: **2 llamadas al modelo** (planificador + sintetizador), sin importar cuántos pasos tenga el plan.

La herramienta es `buscar_faq` sobre la FAQ del curso. Aquí usamos una búsqueda por palabras (como en la clase 2.3) con un retraso artificial que imita una llamada de red. En la clase 3.2 la cambiamos por embeddings.

In [ ]:
# @title ⚙️ La FAQ del curso: muestra incrustada + carga completa si tienes el repo
# --- La FAQ del curso: muestra incrustada + carga completa si tienes el repo ------
# Formato (igual que datos/faq/*.md):  "# Tema"  y luego entradas  "## P: pregunta"  + respuesta.
import pathlib

# Muestra: 14 entradas copiadas de datos/faq/ (mismos archivos y preguntas, para que las citas coincidan).
FAQ_MUESTRA = {
"api-key.md": """# API key de NVIDIA Build

## P: ¿Dónde consigo mi API key de NVIDIA para el curso?
Entra a https://build.nvidia.com e inicia sesión o crea tu cuenta.
Abre tu perfil, ve a **API Keys** y pulsa **Generate API Key**.
Copia la key completa: empieza con `nvapi-`.
Guárdala en un gestor de contraseñas y nunca la pegues en el código.

## P: ¿Dónde guardo mi key y con qué nombre de variable?
Es la misma key `nvapi-` en todo el curso.
En Colab, guárdala en Secrets (icono de llave) como `NVIDIA_API_KEY` y activa el acceso del notebook.
En tu máquina, ponla en el archivo `.env` de la raíz del repo como `NVIDIA_API_KEY` (ese archivo ya está en `.gitignore`).
Para NemoClaw instalado en tu máquina, usa la variable `NVIDIA_INFERENCE_API_KEY`.
""",
"errores-api.md": """# Errores de la API del modelo

## P: ¿Qué significa el error 401 Unauthorized?
Tu key no es válida: está incompleta, tiene espacios o comillas, o la revocaste o regeneraste.
Copia otra vez la key completa (`nvapi-…`) y confirma que tu código lee la variable correcta, por ejemplo con `echo ${NVIDIA_API_KEY:0:6}`, que debe imprimir `nvapi-`.
Reintentar o esperar más no arregla una key inválida.

## P: ¿Por qué me da 403 Forbidden si mi key es válida?
Un 403 de la API significa que tu key no tiene acceso a ese modelo o tu cuenta no tiene cuota.
Prueba con el modelo del curso, `nvidia/nemotron-3.5-lightning-30b-a3b`, y revisa tus créditos en build.nvidia.com.
Si el 403 aparece dentro del sandbox como `CONNECT tunnel failed, response 403`, no es la API: es la política de red de OpenShell.

## P: ¿Qué hago si recibo 429 Too Many Requests?
Mandaste demasiadas solicitudes por minuto.
Espera unos segundos y reintenta una sola vez, respetando el encabezado `Retry-After` si viene.
Pon un límite de pasos al loop del agente y agrega una pausa (`time.sleep()`) entre llamadas en pruebas masivas.
""",
"brev.md": """# Brev y el launchable de NemoClaw

## P: ¿Me cobran si detengo la instancia? ¿Y si la borro?
Encendida (Running), Brev cobra cómputo y disco.
Detenida (Stopped), no cobra cómputo pero sí sigue cobrando el disco, y tus datos se conservan (excepto `/tmp`).
Borrada (Deleted), no cobra nada, pero los datos se pierden para siempre.
Al terminar cada clase, detén la instancia; si no vas a volver en días, bórrala después de guardar tu trabajo en git.

## P: ¿Cómo detengo o borro mi instancia de Brev?
En la consola: brev.nvidia.com, sección GPUs, tu instancia, botón Stop o Delete.
En la CLI: `brev ls` para ver el nombre, `brev stop <instancia>` para detenerla o `brev delete <instancia>` para borrarla, y otra vez `brev ls` para confirmar.
Si `brev stop` responde `does not support stop`, esa instancia solo se puede borrar.

## P: Detuve y volví a encender la instancia y el agente no responde
Tras reiniciar la VM, Docker, el gateway de OpenShell y el sandbox pueden no arrancar solos.
Entra con `brev shell <instancia>` y ejecuta `sudo systemctl start docker`, luego `nemoclaw <sandbox> status`, `nemoclaw <sandbox> start` si quedó detenido y `nemoclaw <sandbox> recover` si el gateway no responde.
""",
"conexion-laboratorio.md": """# Conexión del curso de NVIDIA con el laboratorio

## P: ¿Qué es la "Access session" y de dónde saco la cookie?
Es el valor de la cookie de inicio de sesión de tu launchable.
Abre el launchable en el mismo navegador e inicia sesión.
En DevTools ve a Application, Storage, Cookies, y copia solo el Value completo de la cookie que corresponde: `CF_Authorization` para `brevlab.com`, `_pomerium` para `apps.run.brev.nvidia.com` o `__Host-skybridge-brev-prd` para `gobrev.dev`.
Es una credencial: no la compartas.

## P: Me sale "session expired or unreachable" o "no challenge arrived within 30s"
Tu cookie de acceso venció o el launchable se detuvo.
Confirma en Brev que la instancia está RUNNING, abre el launchable e inicia sesión otra vez, copia un valor nuevo de la cookie y pégalo en Access session.
Lo mismo aplica si ves "Expected JSON but received an HTML page" o "The launchable needs a fresh browser session".
""",
"openshell-politicas.md": """# OpenShell y políticas del sandbox

## P: ¿Qué significa "CONNECT tunnel failed, response 403" dentro del sandbox?
No es un problema de certificados: la red de salida del sandbox está bloqueada por defecto y el proxy de OpenShell rechazó la conexión porque ninguna política la permite.
Mira la línea DENIED con `nemoclaw <sandbox> logs --tail 50`, revisa los presets con `nemoclaw <sandbox> policy list` y agrega el que falte con `nemoclaw <sandbox> policy add <preset>`.

## P: ¿Cómo veo la política activa y agrego un preset como pypi o github?
Ve los presets con `nemoclaw <sandbox> policy list` y la política efectiva completa con `openshell policy get --full <sandbox>`.
Prueba un preset con `nemoclaw <sandbox> policy add pypi --dry-run` y aplícalo con `--yes`; quítalo con `nemoclaw <sandbox> policy remove pypi --yes`.
GitHub no está en la política base: agrégalo a propósito.
Los cambios de red se aplican sin reiniciar.
""",
"seguridad.md": """# Seguridad de keys, cookies y accesos

## P: ¿Dónde guarda NemoClaw mis keys? ¿El agente puede verlas?
NemoClaw registra las keys de los proveedores en el gateway de OpenShell, del lado del host, y los sandboxes nuevos usan proveedores adjuntos que dejan la credencial en el host.
Por eso no pongas keys en `openclaw.json` ni en archivos del workspace: lo que está dentro del sandbox, el agente lo puede leer.
""",
"instalacion-local.md": """# NemoClaw instalado en tu máquina

## P: ¿Qué necesito para instalar NemoClaw en mi computadora?
Mínimo 4 vCPU, 8 GB de RAM y 20 GB de disco libre (recomendado 16 GB y 40 GB).
Necesitas Node.js 22.19 o mayor y Docker funcionando antes de instalar.
Funciona en Ubuntu 24.04, macOS con Apple Silicon y Windows con WSL2; Mac con Intel no está soportado.
Revisa tu máquina con `nemoclaw host probe` e instala con `curl -fsSL https://www.nvidia.com/nemoclaw.sh | bash`.
""",
}

def cargar_faq(carpeta="../../datos/faq"):
    """Devuelve ({archivo: texto}, origen). Usa datos/faq/ si existe (corres dentro del repo); si no, la muestra."""
    p = pathlib.Path(carpeta)
    archivos = [a for a in sorted(p.glob("*.md")) if a.name.lower() != "readme.md"] if p.is_dir() else []
    faq = {a.name: a.read_text(encoding="utf-8") for a in archivos}
    faq = {k: v for k, v in faq.items() if "## P:" in v}
    if faq:
        return faq, f"datos/faq completa ({len(faq)} archivos)"
    return dict(FAQ_MUESTRA), f"muestra incrustada ({len(FAQ_MUESTRA)} archivos)"

def _slug(texto, max_palabras=6):
    return "-".join(re.findall(r"\w+", texto.lower())[:max_palabras])

def trocear(archivo, texto):
    """Un fragmento por cada entrada '## P:'. El tema ('# ...') viaja con cada fragmento."""
    m = re.search(r"^# (.+)$", texto, re.M)
    tema = m.group(1).strip() if m else archivo
    out = []
    for bloque in re.split(r"^## P:\s*", texto, flags=re.M)[1:]:
        pregunta, _, respuesta = bloque.partition("\n")
        pregunta, respuesta = pregunta.strip(), respuesta.strip()
        if pregunta and respuesta and not pregunta.startswith("<"):     # "<...>" = plantilla, no entrada real
            out.append({"id": f"{archivo}#{_slug(pregunta)}", "archivo": archivo, "tema": tema,
                        "pregunta": pregunta, "respuesta": respuesta})
    return out

def fragmentos_de(faq):
    """Trocea todos los archivos y garantiza ids únicos (para poder citarlos)."""
    vistos, out = {}, []
    for archivo, texto in faq.items():
        for f in trocear(archivo, texto):
            n = vistos.get(f["id"], 0)
            vistos[f["id"]] = n + 1
            if n:
                f["id"] += f"-{n + 1}"
            out.append(f)
    return out

FAQ, ORIGEN = cargar_faq()
FRAGMENTOS = fragmentos_de(FAQ)
print(f"FAQ: {ORIGEN} · {len(FRAGMENTOS)} entradas")
for a in FAQ:
    print(f"  {a:18s} {sum(f['archivo'] == a for f in FRAGMENTOS)} entradas")

In [ ]:
import math
from collections import Counter

def _raices(texto):
    """Palabras de 4+ letras recortadas a 5: 'detengo' y 'detener' cuentan como la misma ('deten')."""
    return {w[:5] for w in re.findall(r"\w+", texto.lower()) if len(w) > 3}

_DF = Counter(r for f in FRAGMENTOS for r in _raices(f["pregunta"] + " " + f["respuesta"]))

def _peso(raiz):
    return math.log(1 + len(FRAGMENTOS) / _DF.get(raiz, 1))        # las palabras raras pesan más (IDF)

def buscar_faq(consulta, k=1, demora=0.5):
    """Búsqueda por palabras clave sobre la FAQ (la pregunta pesa doble). `demora` imita una llamada de red."""
    time.sleep(demora)
    q = _raices(consulta)
    def puntaje(f):
        return 2 * sum(map(_peso, q & _raices(f["pregunta"]))) + sum(map(_peso, q & _raices(f["respuesta"])))
    mejores = sorted(FRAGMENTOS, key=puntaje, reverse=True)[:k]
    return [f for f in mejores if puntaje(f) > 0]

MAX_CONSULTAS = 4

def planear(pregunta):
    texto = llm("planificador", pregunta,
                sistema='Divide la pregunta del estudiante en 2 a 4 búsquedas INDEPENDIENTES para la FAQ del curso '
                        '(cortas, en español). Responde SOLO JSON: {"consultas": ["...", "..."]}')
    consultas = extraer_json(texto).get("consultas", [])
    consultas = [c.strip() for c in consultas if isinstance(c, str) and c.strip()]
    consultas = list(dict.fromkeys(consultas))                       # sin duplicados
    if len(consultas) > MAX_CONSULTAS:                               # y con tope: el plan tampoco es confiable
        print(f"   (el plan traía {len(consultas)} consultas; lo recorto a {MAX_CONSULTAS})")
        consultas = consultas[:MAX_CONSULTAS]
    if not consultas:
        raise ValueError(f"El planificador no devolvió consultas útiles: {texto[:120]!r}")
    return consultas

def ejecutar(consultas, paralelo=True):
    if not paralelo:
        return [buscar_faq(c) for c in consultas]
    with ThreadPoolExecutor(max_workers=len(consultas)) as pool:
        return list(pool.map(buscar_faq, consultas))                 # map conserva el orden del plan

def sintetizar(pregunta, consultas, resultados):
    evidencia = "\n".join(
        f"[{i}] {c} → " + (f"[{r[0]['id']}] " + r[0]["respuesta"].replace("\n", " ") if r else "SIN RESULTADOS")
        for i, (c, r) in enumerate(zip(consultas, resultados), 1))
    return llm("sintetizador", f"Pregunta: {pregunta}\n\nEvidencia:\n{evidencia}",
               sistema="Responde al estudiante en 3-6 viñetas usando SOLO la evidencia. Cita cada viñeta con [n]. "
                       "Si falta algo, dilo.")

PREGUNTA = ("Voy a detener mi Brev hasta mañana. ¿Cómo lo detengo, qué pasa con mis archivos "
            "y qué hago si al reanudar el laboratorio me da 401?")
LLAMADAS.clear()
t0 = time.time(); consultas = planear(PREGUNTA);                t_plan = time.time() - t0
t0 = time.time(); resultados = ejecutar(consultas);             t_par = time.time() - t0
t0 = time.time(); respuesta = sintetizar(PREGUNTA, consultas, resultados); t_sint = time.time() - t0

print("PLAN (cada consulta → la entrada que encontró):")
for c, r in zip(consultas, resultados):
    print(f"   {c!r:58} → {r[0]['id'] if r else 'sin resultados'}")
print()
print(respuesta)
print(f"\nllamadas al modelo: {len(LLAMADAS)} {[r for r, _ in LLAMADAS]}")

### Mide la latencia (no la adivines)
¿Cuánto ganamos con el paralelo? Medimos cada fase y comparamos con ejecutar las búsquedas una por una.

In [ ]:
t0 = time.time(); ejecutar(consultas, paralelo=False); t_sec = time.time() - t0

def barra(seg, escala=10):
    return "█" * max(1, round(seg * escala))

print(f"{'planificador':22s} {t_plan:5.2f}s {barra(t_plan)}")
print(f"{'búsquedas (paralelo)':22s} {t_par:5.2f}s {barra(t_par)}")
print(f"{'búsquedas (en serie)':22s} {t_sec:5.2f}s {barra(t_sec)}")
print(f"{'sintetizador':22s} {t_sint:5.2f}s {barra(t_sint)}")
print(f"\nTotal con paralelo: {t_plan + t_par + t_sint:.2f}s · en serie: {t_plan + t_sec + t_sint:.2f}s")
print("Ojo: en vivo, planificador y sintetizador tardan segundos. Mide si el paralelo realmente pesa en el total."
      if not USE_MOCK else "Sin key, las llamadas al modelo tardan lo que simulamos (LATENCIA_MOCK).")

**Cuándo ReWOO NO sirve:** si el paso 2 necesita el resultado del paso 1 (por ejemplo, *"busca el nombre del sandbox del estudiante y luego revisa su política"*), el planificador no puede escribir el paso 2 por adelantado. Usa ReAct, o mete un loop ReAct **dentro** de una etapa del flujo.

## 4 · Fan-out / fan-in: varios especialistas a la vez, con una rama que falla
El reporte *"Me sale 401 en el laboratorio después de reanudar Brev"* toca **varios** temas. En vez de elegir uno, lanzamos tres especialistas en paralelo con el **mismo** reporte (fan-out) y una última llamada **une** sus hallazgos (fan-in).

Con `as_completed` recogemos cada rama por separado. Si una falla, **guardamos el error y seguimos**. Es lo contrario de `Promise.all` en JavaScript (el que usa la lección de NVIDIA), que rechaza todo en cuanto una rama falla.

In [ ]:
REPORTE = "Me sale 401 en el laboratorio después de reanudar Brev. Ayer funcionaba."
RAMAS = ["api_key", "brev", "seguridad"]

def rama(nombre, reporte, falla=None):
    t0 = time.time()
    if nombre == falla:
        time.sleep(0.2)
        raise TimeoutError("el especialista no respondió a tiempo (simulado)")
    return ESPECIALISTAS[nombre](reporte), time.time() - t0

def fan_out(reporte, ramas=RAMAS, falla=None, max_workers=3, fn=rama):
    hallazgos, errores, tiempos = {}, {}, {}
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futuros = {pool.submit(fn, n, reporte, falla): n for n in ramas}
        for f in as_completed(futuros):
            n = futuros[f]
            try:
                hallazgos[n], tiempos[n] = f.result()
            except Exception as e:                        # una rama caída NO tumba a las demás
                errores[n] = f"{type(e).__name__}: {e}"
    return hallazgos, errores, tiempos, time.time() - t0

def unir(reporte, hallazgos, errores):
    cuerpo = "\n".join(f"[{n}] {t}" for n, t in sorted(hallazgos.items()))
    if errores:
        cuerpo += "\n" + "\n".join(f"[{n}] NO DISPONIBLE ({e})" for n, e in errores.items())
    return llm("fusion", f"Reporte: {reporte}\n\nHallazgos:\n{cuerpo}",
               sistema="Escribe UNA respuesta para el estudiante: causa más probable y pasos. "
                       "Si una revisión faltó, dilo y advierte que la respuesta puede estar incompleta.")

LLAMADAS.clear()
hallazgos, errores, tiempos, t_total = fan_out(REPORTE, falla="seguridad")
for n in RAMAS:
    estado = f"{tiempos[n]:.2f}s {barra(tiempos[n])}" if n in tiempos else f"FALLÓ → {errores[n]}"
    print(f"  {n:10s} {estado}")
print(f"  reloj total del fan-out: {t_total:.2f}s (suma de ramas: {sum(tiempos.values()):.2f}s)\n")
print(unir(REPORTE, hallazgos, errores))
print("\nllamadas al modelo:", len(LLAMADAS))

## 5 · Cuándo NO dividir
Cada frontera nueva cuesta: **una llamada más**, más latencia, un lugar más donde el modelo puede devolver basura y una traza más difícil de leer. Compara el fan-out con **una sola llamada** a un especialista generalista:

In [ ]:
generalista = crear_especialista("generalista", BASE + "Cubres instalación, API key, Brev y seguridad.")

LLAMADAS.clear(); t0 = time.time()
r1 = generalista(REPORTE)
t_una, n_una = time.time() - t0, len(LLAMADAS)

LLAMADAS.clear(); t0 = time.time()
h, e, _, _ = fan_out(REPORTE)
r2 = unir(REPORTE, h, e)
t_fan, n_fan = time.time() - t0, len(LLAMADAS)

print(f"una llamada : {n_una} llamada(s), {t_una:.2f}s\n  {r1[:160]}\n")
print(f"fan-out     : {n_fan} llamada(s), {t_fan:.2f}s\n  {r2[:160]}")

| Divide (router, ReWOO, fan-out) cuando… | NO dividas cuando… |
|---|---|
| Las ramas son **independientes** y su unión es clara | El paso siguiente depende del anterior |
| Cada rama necesita un prompt o herramientas **distintas** | Una sola llamada ya responde bien |
| El paralelo **ahorra tiempo medido** | El tiempo lo dominan las llamadas al modelo, no las herramientas |
| Necesitas aislar fallas (una rama cae, las demás siguen) | Leer y depurar la traza es más importante que la velocidad |

**Regla:** empieza con una llamada o un loop. Divide solo cuando midas que la división mejora la calidad o el tiempo.

## 🧯 Cuando algo falla
### a) El modelo devuelve JSON roto o una etiqueta que no existe
Pasa en la vida real: `finish_reason=length` corta el JSON a la mitad, el modelo agrega prosa o inventa una categoría. Prueba tu validador con salidas malas reales:

In [ ]:
SALIDAS_MALAS = [
    '{"categoria": "facturacion", "prioridad": "alta", "resumen": "reembolso"}',   # etiqueta inventada
    'Claro! {"categoria": "brev", "prioridad": "alta", "resumen": "401"}',          # prosa alrededor: se rescata
    '{"categoria": "Brev", "prioridad": "alta", "resumen": "401"}',                 # mayúscula: NO es lo mismo
    '{"categoria": "brev", "prio',                                                  # cortado por max_tokens
    "No estoy seguro de la categoría.",                                             # sin JSON
]
for s in SALIDAS_MALAS:
    try:
        errores = validar_triage(extraer_json(s))
    except (ValueError, json.JSONDecodeError) as e:
        errores = [f"JSON ilegible ({type(e).__name__})"]
    print(f"{s[:58]!r:62} -> {'OK' if not errores else errores}")

> ¿Normalizar `"Brev"` a `"brev"`? Puedes hacer limpiezas **triviales** (espacios, mayúsculas). Lo que **nunca** debes hacer es mapear sinónimos inventados (`"facturacion"` → `"brev"`): eso es adivinar, y un router que adivina manda tickets al lugar equivocado sin avisar.

### b) Un 429 en una rama paralela
Lanzar muchas llamadas a la vez es la forma más rápida de chocar con el límite por minuto. Simulamos que la rama `seguridad` recibe un **429** y vemos el diagnóstico en español (la función `diagnostico()` del curso):

In [ ]:
intentos_por_rama = {}

def rama_con_429(nombre, reporte, falla=None):
    """La rama 'seguridad' recibe un 429 en su primer intento (simulado; en vivo pasa al lanzar ráfagas)."""
    intentos_por_rama[nombre] = intentos_por_rama.get(nombre, 0) + 1
    t0 = time.time()
    if nombre == "seguridad" and intentos_por_rama[nombre] == 1:
        raise RuntimeError(diagnostico(ErrorSimulado(429, '{"status": 429, "title": "Too Many Requests"}')))
    return ESPECIALISTAS[nombre](reporte), time.time() - t0

h, e, _, _ = fan_out(REPORTE, fn=rama_con_429)
print("ramas OK:", sorted(h))
print("errores :", {k: v.splitlines()[0][:110] for k, v in e.items()})

In [ ]:
def con_reintento(fn, intentos=3, espera=0.5):
    """Envuelve una rama: reintenta SOLO ante 429, con espera creciente (backoff)."""
    def envuelta(*args):
        for i in range(1, intentos + 1):
            try:
                return fn(*args)
            except RuntimeError as e:
                if "429" not in str(e)[:5] or i == intentos:
                    raise                                    # 401, 404... no se arreglan reintentando
                print(f"   429 en {args[0]} (intento {i}): espero {espera * i:.1f}s y reintento")
                time.sleep(espera * i)
    return envuelta

intentos_por_rama.clear()
h, e, _, _ = fan_out(REPORTE, fn=con_reintento(rama_con_429), max_workers=2)   # además, menos hilos
print("ramas OK:", sorted(h), "| errores:", e)

**Qué aprendiste del fallo:** el reintento va **dentro de la rama** (cada rama decide si su error es reintentable) y el fan-out solo registra lo que finalmente falló. Un 401 o un 404 **no** se reintentan: no se arreglan esperando.

## 🚀 Avance del proyecto · Soporte NemoClaw v3.1
Tu agente ya **clasifica** cada ticket y lo manda al especialista correcto. Si el ticket es de prioridad alta, además corre el fan-out para no quedarse con una sola mirada.

In [ ]:
def soporte_nemoclaw(ticket):
    t = clasificar(ticket)
    if t["categoria"] == "otro" or t["prioridad"] != "alta":
        return t["categoria"], ESPECIALISTAS[t["categoria"]](ticket)
    hallazgos, errores, _, _ = fan_out(ticket, fn=con_reintento(rama))
    return t["categoria"] + " + fan-out", unir(ticket, hallazgos, errores)

for ticket in ["¿Necesito GPU para el curso?", REPORTE]:
    ruta, respuesta = soporte_nemoclaw(ticket)
    print(f"[{ruta}] {ticket}\n{respuesta}\n")

En la clase 3.2 los especialistas dejan de responder "de memoria": les damos **RAG** sobre la FAQ del curso, con citas.

## 🏁 Reto
Agrega la categoría **`costos`** (dudas de créditos y precio de Brev; en la clase 2.3 hiciste `estimar_costo_brev`):
1. Agrégala a `CATEGORIAS`, al prompt `SISTEMA_TRIAGE` y a `ESPECIALISTAS`.
2. Escribe 3 tickets de costos y 1 ambiguo.

**Criterio de éxito:** los 3 tickets de costos llegan a `costos`, el de reembolso sigue cayendo en `otro` por validación (no por suerte) y ningún ticket rompe el pipeline. *Pista:* sin key, agrega una regla al mock o prueba directamente con tu key.

📎 [Guía de la clase](README.md) · [Diapositivas](slides.html) · [FAQ del curso](../../FAQ.md)